# 07. PSI Monitoring

Goal: measure population stability of the frozen scorecard (`models/scorecard_pipeline.json`).

- Expected (baseline) population: training set
- Observed populations: (a) validation set, expected to show PSI near zero (sanity baseline), (b) `application_test.csv`, a genuinely unseen population used as a proxy for incoming applicants
- Note: the dataset has no calendar dates, so (b) is not true temporal drift. It is the best available proxy for a new population.
- PSI is computed per scorecard characteristic and on the final score, always using the frozen training bins (never recalculated).

In [1]:
# Imports and paths
import json
import numpy as np
import pandas as pd

DATA_RAW = '../data/raw/'
DATA_PROC = '../data/processed/'
PIPELINE_PATH = '../models/scorecard_pipeline.json'

In [2]:
# Load application_test, defragment first, then replicate the 02 treatment exactly (including .round(1))
app_test = pd.read_csv(DATA_RAW + 'application_test.csv').copy()
app_test['DAYS_EMPLOYED_ANOM'] = (app_test['DAYS_EMPLOYED'] == 365243).astype('int8')
app_test['DAYS_EMPLOYED'] = app_test['DAYS_EMPLOYED'].replace(365243, np.nan)
app_test['AGE_YEARS'] = (-app_test['DAYS_BIRTH'] / 365).round(1)
app_test['YEARS_EMPLOYED'] = (-app_test['DAYS_EMPLOYED'] / 365).round(1)
app_test['YEARS_REGISTRATION'] = (-app_test['DAYS_REGISTRATION'] / 365).round(1)
app_test['YEARS_ID_PUBLISH'] = (-app_test['DAYS_ID_PUBLISH'] / 365).round(1)
app_test['YEARS_LAST_PHONE_CHANGE'] = (-app_test['DAYS_LAST_PHONE_CHANGE'] / 365).round(1)

# Merge with the saved bureau aggregation and check bureau coverage
bureau_agg = pd.read_csv(DATA_PROC + 'bureau_agg.csv')
app_test = app_test.merge(bureau_agg, on='SK_ID_CURR', how='left')
no_bureau = app_test['BUREAU_CREDIT_COUNT'].isnull().mean()
print(app_test.shape, f'no bureau history: {no_bureau:.1%} (train was 14.3%)')

(48744, 155) no bureau history: 13.2% (train was 14.3%)


In [3]:
# Build the 17-variable raw feature frames for train and application_test
FINAL_17_VARS = ['EXT_SOURCE_2', 'EXT_SOURCE_3', 'NAME_EDUCATION_TYPE_BINNED', 'EXT_SOURCE_1',
                 'AMT_CREDIT', 'CODE_GENDER', 'YEARS_EMPLOYED', 'ORGANIZATION_TYPE_BINNED',
                 'FLOORSMAX_AVG', 'BUREAU_CREDIT_ACTIVE_ACTIVE_COUNT',
                 'BUREAU_AMT_CREDIT_SUM_LIMIT_MEAN', 'OCCUPATION_TYPE_BINNED',
                 'REGION_POPULATION_RELATIVE', 'DAYS_ID_PUBLISH',
                 'BUREAU_AMT_CREDIT_SUM_DEBT_MEAN', 'DAYS_LAST_PHONE_CHANGE',
                 'BUREAU_DAYS_CREDIT_MEAN']

# Same RAW_SOURCE as 06: _BINNED variables are fed from the raw Kaggle columns
RAW_SOURCE = {v: v for v in FINAL_17_VARS}
RAW_SOURCE['NAME_EDUCATION_TYPE_BINNED'] = 'NAME_EDUCATION_TYPE'
RAW_SOURCE['ORGANIZATION_TYPE_BINNED'] = 'ORGANIZATION_TYPE'
RAW_SOURCE['OCCUPATION_TYPE_BINNED'] = 'OCCUPATION_TYPE'

def build_X(df):
    return pd.DataFrame({v: df[src] for v, src in RAW_SOURCE.items()})

train_bureau = pd.read_csv(DATA_PROC + 'train_bureau.csv')
X_all = build_X(train_bureau)
X_test = build_X(app_test)
print(X_all.shape, X_test.shape)
print(X_test.isnull().mean().round(3).sort_values(ascending=False).head(8))

(307511, 17) (48744, 17)
FLOORSMAX_AVG                       0.478
EXT_SOURCE_1                        0.421
OCCUPATION_TYPE_BINNED              0.320
BUREAU_AMT_CREDIT_SUM_LIMIT_MEAN    0.219
YEARS_EMPLOYED                      0.190
EXT_SOURCE_3                        0.178
BUREAU_AMT_CREDIT_SUM_DEBT_MEAN     0.153
BUREAU_DAYS_CREDIT_MEAN             0.132
dtype: float64


In [4]:
# Compare missing rates: full train_bureau vs application_test (descriptive, not PSI yet)
miss_cmp = pd.DataFrame({
    'train': X_all.isnull().mean(),
    'test': X_test.isnull().mean(),
})
miss_cmp['diff_pp'] = ((miss_cmp['test'] - miss_cmp['train']) * 100).round(1)
print(miss_cmp.round(3).sort_values('diff_pp', key=abs, ascending=False))

                                   train   test  diff_pp
EXT_SOURCE_1                       0.564  0.421    -14.3
EXT_SOURCE_3                       0.198  0.178     -2.0
FLOORSMAX_AVG                      0.498  0.478     -1.9
BUREAU_AMT_CREDIT_SUM_DEBT_MEAN    0.167  0.153     -1.5
BUREAU_CREDIT_ACTIVE_ACTIVE_COUNT  0.143  0.132     -1.1
BUREAU_DAYS_CREDIT_MEAN            0.143  0.132     -1.1
YEARS_EMPLOYED                     0.180  0.190      1.0
BUREAU_AMT_CREDIT_SUM_LIMIT_MEAN   0.212  0.219      0.8
OCCUPATION_TYPE_BINNED             0.313  0.320      0.7
EXT_SOURCE_2                       0.002  0.000     -0.2
CODE_GENDER                        0.000  0.000      0.0
NAME_EDUCATION_TYPE_BINNED         0.000  0.000      0.0
AMT_CREDIT                         0.000  0.000      0.0
ORGANIZATION_TYPE_BINNED           0.000  0.000      0.0
REGION_POPULATION_RELATIVE         0.000  0.000      0.0
DAYS_ID_PUBLISH                    0.000  0.000      0.0
DAYS_LAST_PHONE_CHANGE         

In [5]:
# EXT_SOURCE_1: distribution of non-missing values, train vs test
es1 = pd.DataFrame({
    'train': X_all['EXT_SOURCE_1'].dropna().describe(),
    'test': X_test['EXT_SOURCE_1'].dropna().describe(),
})
print(es1.round(4))

             train        test
count  134133.0000  28212.0000
mean        0.5021      0.5012
std         0.2111      0.2051
min         0.0146      0.0135
25%         0.3340      0.3437
50%         0.5060      0.5068
75%         0.6751      0.6660
max         0.9627      0.9391


In [6]:
# Inspect the structure of the frozen pipeline JSON (one numeric, one categorical variable)
with open(PIPELINE_PATH) as f:
    pipeline = json.load(f)
print(list(pipeline.keys()))
first_vars = pipeline['variables'] if 'variables' in pipeline else pipeline
for name in ['EXT_SOURCE_1', 'BUREAU_AMT_CREDIT_SUM_LIMIT_MEAN', 'CODE_GENDER']:
    print(name, json.dumps(first_vars[name], indent=1)[:700], '\n')

['metadata', 'factor', 'offset', 'intercept', 'base_points_per_characteristic', 'variables']
EXT_SOURCE_1 {
 "type": "numeric",
 "coefficient": -0.5263759646745365,
 "edges": [
  -1e+18,
  0.212,
  0.296,
  0.368,
  0.437,
  0.505,
  0.573,
  0.64,
  0.71,
  0.787,
  1e+18
 ],
 "bin_woe": [
  -0.8754785826787144,
  -0.40537622349824226,
  -0.13521251952453248,
  -0.04927108886022368,
  0.18269706680936673,
  0.32596067190564304,
  0.47672057073110596,
  0.6600302662824113,
  0.9081534191024493,
  1.1977890654497467
 ],
 "zero_woe": null,
 "missing_woe": -0.0591843975525508
} 

BUREAU_AMT_CREDIT_SUM_LIMIT_MEAN {
 "type": "numeric",
 "coefficient": -0.2410898790562269,
 "edges": [
  -1e+18,
  307.883,
  1307.523,
  3190.582,
  6249.647,
  10775.642,
  17358.08,
  26935.632,
  42000.0,
  74358.639,
  1e+18
 ],
 "bin_woe": [
  -0.708743383857389,
  -0.6007791457534276,
  -0.5141929286079838,
  -0.3263419473674939,
  -0.3604001881035479,
  -0.14101469187295726,
  -0.0479642227168859,
  0.14

In [7]:
# Vectorized WoE from the frozen JSON, mirroring score_client rules
def assign_bins(s, spec):
    if spec['type'] == 'numeric':
        idx = pd.cut(s, bins=spec['edges'], labels=False)
        lab = pd.Series(
            np.where(idx.notnull(), 'bin_' + idx.fillna(-1).astype(int).astype(str).str.zfill(2), 'Missing'),
            index=s.index, dtype='object')
        if spec.get('zero_woe') is not None:
            lab = lab.mask(s == 0, 'Zero')
        return lab.mask(s.isnull(), 'Missing')
    cw, cmap = spec['category_woe'], spec.get('category_map')
    c = s.astype('object').where(s.notnull(), 'Missing')
    if cmap is not None:
        fallback = 'Other_grouped' if 'Other_grouped' in cw else ('Missing' if 'Missing' in cw else 'Unmapped')
        c = c.map(cmap).fillna(fallback)
    return c.where(c.isin(list(cw)), 'Unmapped')

def woe_from_labels(lab, spec):
    if spec['type'] == 'numeric':
        m = {f'bin_{i:02d}': w for i, w in enumerate(spec['bin_woe'])}
        m['Zero'] = spec.get('zero_woe')
        m['Missing'] = spec.get('missing_woe')
        return lab.map(m).astype(float)
    m = dict(spec['category_woe'])
    m['Unmapped'] = 0.0
    return lab.map(m).astype(float)

def score_vectorized(X, pipeline):
    V = pipeline['variables']
    lab = pd.DataFrame({v: assign_bins(X[v], sp) for v, sp in V.items()})
    woe = pd.DataFrame({v: woe_from_labels(lab[v], sp) for v, sp in V.items()})
    coef = pd.Series({v: sp['coefficient'] for v, sp in V.items()})
    pts = pipeline['base_points_per_characteristic'] + (-pipeline['factor']) * woe.mul(coef)
    log_odds = woe.mul(coef).sum(axis=1) + pipeline['intercept']
    out = pd.DataFrame({'score': pts.sum(axis=1).round(2),
                        'pd': (1 / (1 + np.exp(-log_odds))).round(4)})
    return lab, woe, out

In [8]:
# Validate vectorized scoring against score_client on a random sample of the real data
import sys
sys.path.append('../src')
from scorecard_utils import score_client

with open(PIPELINE_PATH) as f:
    pipeline = json.load(f)

sample = X_all.sample(1000, random_state=7)
_, _, vec = score_vectorized(sample, pipeline)

ref = pd.DataFrame(
    [(i, o['score'], o['probability_default'])
     for i, r in sample.iterrows()
     for o in [score_client(r.to_dict(), pipeline)]],
    columns=['idx', 'score_ref', 'pd_ref']).set_index('idx')

cmp = vec.join(ref)
bad = cmp[((cmp['score'] - cmp['score_ref']).abs() > 0.011) | ((cmp['pd'] - cmp['pd_ref']).abs() > 2e-4)]
print(len(bad), 'mismatches of', len(cmp))
if len(bad):
    print('share with OCCUPATION_TYPE_BINNED null:', sample.loc[bad.index, 'OCCUPATION_TYPE_BINNED'].isnull().mean())

0 mismatches of 1000
